What is the coverage of the data  on pools?
number of pools per state and constituency?
number of reviews per pools available in the dataset 

In [ ]:
import pandas as pd
import numpy as np



In [ ]:
# dataset of all collected reviews
reviews_df= pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/all_collected_reviews_25112025.parquet")
reviews_df.info()

In [ ]:
pools_df = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/swimmingpool_list_to_use_25112025.csv")
pools_df.drop(columns=['status', 'cluster_id', 'index_right', 'Unnamed: 54', 'Unnamed: 55', 'match_score', 'match_type'], inplace=True)
pools_df.info()

## Data distribution of  all reviews collected

In [ ]:
### aggregating the reviews by placeid
# Count collected reviews per place
reviews_per_place = (
    reviews_df
    .groupby("place_id", as_index=False)
    .agg(
        reviews_collected=("review_id", "count")
    )
)
reviews_per_place.info()

In [ ]:
#### merge with pools df
coverage_df = (
    pools_df
    .merge(reviews_per_place, on="place_id", how="left")
    .assign(
        reviews_collected=lambda d: d["reviews_collected"].fillna(0).astype(int),
        coverage_ratio=lambda d: np.where(
            d["total_reviews"] > 0,
            d["reviews_collected"] / d["total_reviews"],
            np.nan
        )
    )
)
coverage_df.info()

In [ ]:
## pool per constitunency
pools_by_constituency = (
    coverage_df
    .groupby("WKR_NAME", as_index=False)
    .agg(
        n_pools=("place_id", "count")
    )
    .assign(
        pool_share=lambda d: d["n_pools"] / d["n_pools"].sum()
    )
    .sort_values("n_pools", ascending=False)
)
pools_by_constituency.info()

In [ ]:
#number of pools per state
pools_by_state = (
    coverage_df
    .groupby("LAND_NAME", as_index=False)
    .agg(
        n_pools=("place_id", "count")
    )
    .assign(
        pool_share=lambda d: d["n_pools"] / d["n_pools"].sum()
    )
    .sort_values("n_pools", ascending=False)
)

pools_by_state.info()

In [ ]:
# Review distribution across pools (global)

review_distribution_summary = (
    coverage_df
    .agg(
        n_pools=("place_id", "count"),
        total_reviews_collected=("reviews_collected", "sum"),
        mean_reviews=("reviews_collected", "mean"),
        median_reviews=("reviews_collected", "median"),
        max_reviews=("reviews_collected", "max"),
        mean_coverage=("coverage_ratio", "mean"),
        median_coverage=("coverage_ratio", "median")
    )
)
review_distribution_summary


In [ ]:
#review coverage by constituency
coverage_by_constituency = (
    coverage_df
    .groupby("WKR_NAME", as_index=False)
    .agg(
        n_pools=("place_id", "count"),
        total_reviews=("reviews_collected", "sum"),
        mean_reviews_per_pool=("reviews_collected", "mean"),
        median_reviews_per_pool=("reviews_collected", "median"),
        mean_coverage=("coverage_ratio", "mean"),
        median_coverage=("coverage_ratio", "median"),
        fully_covered_share=("coverage_ratio", lambda x: (x >= 0.8).mean())
    )
    .sort_values("n_pools", ascending=False)
)

coverage_by_constituency.info()

In [ ]:
#review coverage by state
coverage_by_state = (
    coverage_df
    .groupby("LAND_NAME", as_index=False)
    .agg(
        n_pools=("place_id", "count"),
        total_reviews=("reviews_collected", "sum"),
        mean_reviews_per_pool=("reviews_collected", "mean"),
        median_reviews_per_pool=("reviews_collected", "median"),
        mean_coverage=("coverage_ratio", "mean"),
        median_coverage=("coverage_ratio", "median"),
        fully_covered_share=("coverage_ratio", lambda x: (x >= 0.8).mean())
    )
    .sort_values("n_pools", ascending=False)
)
coverage_by_state.info()

In [ ]:
#Diagnostic columns for interpretation (VERY useful)
coverage_df = coverage_df.assign(
    review_volume_bin=pd.cut(
        coverage_df["total_reviews"],
        bins=[0, 10, 50, 100, 250, 500, np.inf],
        labels=["≤10", "11–50", "51–100", "101–250", "251–500", "500+"]
    ),
    coverage_class=pd.cut(
        coverage_df["coverage_ratio"],
        bins=[0, 0.25, 0.5, 0.75, 0.95, 1.01],
        labels=["<25%", "25–50%", "50–75%", "75–95%", "≈100%"]
    )
)
coverage_df.info()

#### Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")


In [ ]:
#Distribution of reviews per pool (skew check)
# Purpose:
# Shows whether a small number of pools dominate the review volume.

plt.figure(figsize=(8, 5))

sns.histplot(
    coverage_df["reviews_collected"],
    bins=50,
    log_scale=(True, False)
)

plt.xlabel("Reviews collected per pool (log scale)")
plt.ylabel("Number of pools")
plt.title("Distribution of Reviews per Pool")

plt.tight_layout()
plt.show()

In [ ]:
#Distribution of coverage ratios
# Purpose:
# Distinguishes partial collection from complete collection.

plt.figure(figsize=(8, 5))
sns.histplot(
    coverage_df["coverage_ratio"].dropna(),
    bins=30
)

plt.xlabel("Coverage ratio (collected / available reviews)")
plt.ylabel("Number of pools")
plt.title("Distribution of Review Coverage Across Pools")

plt.tight_layout()
plt.show()


In [ ]:
# #Collected vs available reviews (key diagnostic plot)
# Purpose:
# Visually separates:
# low-review but complete pools
# high-review but partially collected pools
# This is the most important figure.

plt.figure(figsize=(7, 7))

sns.scatterplot(
    data=coverage_df,
    x="total_reviews",
    y="reviews_collected",
    alpha=0.4,
    s=30
)

max_reviews = coverage_df["total_reviews"].max()

plt.plot(
    [0, max_reviews],
    [0, max_reviews],
    linestyle="--",
    color="red",
    label="Full coverage (y = x)"
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Total reviews available (log)")
plt.ylabel("Reviews collected (log)")
plt.title("Collected vs Available Reviews per Pool")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
#Pools per state (coverage of places)
#Purpose:
#Shows spatial representativeness of pools.
plt.figure(figsize=(10, 5))

sns.barplot(
    data=coverage_by_state,
    x="LAND_NAME",
    y="n_pools"
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("State")
plt.ylabel("Number of pools")
plt.title("Number of Pools per State")

plt.tight_layout()
plt.show()



In [ ]:
#Mean coverage ratio per state
#Purpose:
#Shows data quality differences across space.

plt.figure(figsize=(10, 5))

sns.barplot(
    data=coverage_by_state,
    x="LAND_NAME",
    y="mean_coverage"
)

plt.axhline(0.8, linestyle="--", color="red", label="80% coverage threshold")

plt.xticks(rotation=45, ha="right")
plt.xlabel("State")
plt.ylabel("Mean coverage ratio")
plt.title("Average Review Coverage per State")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
#6. Constituency-level coverage vs number of pools
# Purpose:
# Identifies constituencies that may bias results due to:
# few pools
# low coverage

plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=coverage_by_constituency,
    x="n_pools",
    y="mean_coverage",
    alpha=0.6
)

plt.xlabel("Number of pools in constituency")
plt.ylabel("Mean coverage ratio")
plt.title("Coverage vs Pool Count by Constituency")

plt.tight_layout()
plt.show()


In [ ]:
#Coverage class composition (stacked bar)
# Purpose:
# Shows how much of your dataset is fully vs partially collected.


coverage_class_dist = (
    coverage_df
    .groupby("coverage_class")
    .size()
    .reset_index(name="n_pools")
    .assign(share=lambda d: d["n_pools"] / d["n_pools"].sum())
)

plt.figure(figsize=(7, 5))

sns.barplot(
    data=coverage_class_dist,
    x="coverage_class",
    y="share"
)

plt.xlabel("Coverage class")
plt.ylabel("Share of pools")
plt.title("Coverage Completeness of Pools")

plt.tight_layout()
plt.show()


In [ ]:
#(Optional but powerful) Review volume × coverage heatmap
# Purpose:
# Compact summary of where partial coverage concentrates.

heatmap_data = (
    coverage_df
    .groupby(["review_volume_bin", "coverage_class"])
    .size()
    .unstack(fill_value=0)
)

plt.figure(figsize=(9, 6))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Coverage class")
plt.ylabel("Review volume bin")
plt.title("Review Volume vs Coverage Completeness")

plt.tight_layout()
plt.show()
